In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

output_dir = Path(os.environ['OUTPUT_DATA_DIR'])
df_agg = pd.read_csv(output_dir / 'fmri_stats.tsv', sep='\t')
df_sub = pd.read_csv(output_dir / 'fmri_stats_per_subject.tsv', sep='\t')

In [ ]:
# Figure 1: total hours of fMRI per dataset
datasets = df_agg['dataset'].tolist()
values = df_agg['total_duration_h'].tolist()

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.bar(range(len(datasets)), values)
ax.set_xticks(range(len(datasets)))
ax.set_xticklabels(datasets, rotation=45, ha='right')
ax.set_ylabel('Hours')
ax.set_title('CNeuroMod: total fMRI acquisition time per dataset', fontweight='bold')

out_path = output_dir / 'figure_fmri_total_hours.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

In [ ]:
# Figure 2: min / avg / max of total_duration_h across subjects with data, per dataset
# Only include subjects who have total_runs > 0 (i.e. have data in the dataset)
has_data = df_sub[df_sub['total_runs'] > 0].dropna(subset=['total_duration_h'])

grouped = has_data.groupby('dataset')['total_duration_h']
stats = grouped.agg(['min', 'mean', 'max']).reset_index()
stats.columns = ['dataset', 'min', 'mean', 'max']

datasets = stats['dataset'].tolist()
n = len(datasets)
x = np.arange(n)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.bar(x - width, stats['min'],  width, label='Min')
ax.bar(x,          stats['mean'], width, label='Avg')
ax.bar(x + width, stats['max'],  width, label='Max')

ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha='right')
ax.set_ylabel('Hours')
ax.set_title('CNeuroMod: total fMRI hours per dataset — min / avg / max across subjects with data',
             fontweight='bold')
ax.legend()

out_path = output_dir / 'figure_fmri_hours_min_avg_max.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

In [ ]:
# Figure 3: total acquisition time per dataset per subject (one panel per subject)
subjects = sorted(df_sub['subject'].unique())
datasets = df_agg['dataset'].tolist()
total_label = 'TOTAL'
x_labels = datasets + [total_label]
n_bars = len(x_labels)

fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True, sharey=True)
axes_flat = axes.flatten()

for ax, subject in zip(axes_flat, subjects):
    sub_df = df_sub[df_sub['subject'] == subject].set_index('dataset')
    values = []
    for ds in datasets:
        if ds in sub_df.index and sub_df.loc[ds, 'total_runs'] > 0:
            values.append(sub_df.loc[ds, 'total_duration_h'])
        else:
            values.append(0.0)
    total = sum(v for v in values if v > 0)
    values_with_total = values + [total]

    colors = ['#1f77b4'] * len(datasets) + ['#ff7f0e']
    ax.bar(range(n_bars), values_with_total, color=colors)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    ax.set_xticks(range(n_bars))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(subject, fontweight='bold')
    ax.set_ylabel('Hours')

fig.suptitle('CNeuroMod: total fMRI acquisition time per dataset per subject', fontweight='bold', fontsize=13)

out_path = output_dir / 'total_acquisition_time_per_dataset_per_subject.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')
